# FATO_LEITOS

Fato de leitos, um **snapshot periódico**. **Grão:** competência (mês) × estabelecimento.

- **Medidas:** `QTD_LEITOS_EXISTENTES` e `QTD_LEITOS_SUS`. São **semiaditivas**: somam
  entre estabelecimentos, mas não ao longo do tempo (no período, use a média mensal).
- Usada no cálculo da taxa de ocupação (pergunta P04).

**Estratégia de carga:** full (`TRUNCATE` + `INSERT /*+ APPEND */`), idempotente.

In [ ]:
import sys
sys.path.append("..")
from dw_utils import recarregar, consultar, coletar_estatisticas

In [ ]:
SQL = """
INSERT /*+ APPEND */ INTO FATO_LEITOS
    (SK_TEMPO_COMPETENCIA, SK_ESTABELECIMENTO, QTD_LEITOS_EXISTENTES, QTD_LEITOS_SUS)
SELECT NVL(t.SK_TEMPO, -1),
       NVL(e.SK_ESTABELECIMENTO, -1),
       SUM(s.QTD_LEITOS_EXISTENTES),
       SUM(s.QTD_LEITOS_SUS)
FROM SILVER_LEITOS s
LEFT JOIN DIM_TEMPO t           ON t.DT_COMPLETA = s.DT_COMPETENCIA
LEFT JOIN DIM_ESTABELECIMENTO e ON e.COD_CNES = s.COD_CNES
GROUP BY NVL(t.SK_TEMPO, -1), NVL(e.SK_ESTABELECIMENTO, -1)
"""

In [ ]:
recarregar("FATO_LEITOS", SQL, origem="SILVER_LEITOS")
coletar_estatisticas("FATO_LEITOS")

In [ ]:
consultar("SELECT * FROM FATO_LEITOS FETCH FIRST 10 ROWS ONLY")